## Imports

In [0]:
exec(open("/Workspace/Users/4020arjumandrashid@gmail.com/NASA-EONET/00_config").read())

In [0]:
%sql
USE CATALOG nasa_eonet;
USE SCHEMA events;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType
from pyspark.sql.functions import col, expr, to_timestamp, array_min, array_max, transform, from_json
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, DoubleType
from pyspark.databricks.sql import functions as dbf

In [0]:
!pip install spacy

In [0]:
import spacy
from collections import Counter
from string import punctuation

In [0]:
!python -m spacy download en_core_web_sm

## Helpers

In [0]:
nlp = spacy.load("en_core_web_sm")

def get_hotwords(text):
    result = []
    pos_tag = ['PROPN', 'ADJ', 'NOUN']
    doc = nlp(text.lower())
    for token in doc:
        if(token.text in nlp.Defaults.stop_words or token.text in punctuation):
            continue
        if(token.pos_ in pos_tag):
            result.append(token.text)
    return result

## dim_categories

In [0]:
cat_df = spark.table(
    "nasa_eonet.events_silver.categories"
)

In [0]:
get_hotwords_udf = F.udf(get_hotwords, ArrayType(StringType()))
cat_df = cat_df.withColumn('hotwords', get_hotwords_udf(F.col('description')))

In [0]:
cat_df.show()

In [0]:
cat_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_gold.dim_categories")

## dim_sources

In [0]:
src_df = spark.table(
    "nasa_eonet.events_silver.sources"
)

In [0]:
src_df.show()

In [0]:
src_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_gold.dim_sources")

## dim_events

In [0]:
events_df = spark.table(
    "nasa_eonet.events_silver.events_geo"
)

In [0]:
events_df.count()

In [0]:
events_df.show(5)

dim_points

It is not possible to fetch distinct rows of GEOMETRY data type in spark, as it does not know how to hash the spatial data type.

In [0]:
points_df = spark.sql(
    """
    select avg_point from nasa_eonet.events_silver.events_geo
    """
)

In [0]:
points_df = points_df.withColumn(
    "point_id", F.expr("uuid()")
)

In [0]:
from pyspark.sql import Window

events_df = events_df.join(
    points_df,
    dbf.st_equals(events_df["avg_point"], points_df["avg_point"]),
    how = "left"
).dropDuplicates(["point_id"])

In [0]:
points_df.select(
    "point_id", F.col("avg_point").alias("point_geometry")
).write.format("delta").mode("overwrite").saveAsTable("nasa_eonet.events_gold.dim_points")

dim_geometry

In [0]:
geometry_df = spark.sql(
    """
    select `ISO3166-1-Alpha-3`, geometry, name from nasa_eonet.events_silver.events_geo
    """
)

In [0]:
geometry_df.show()

In [0]:
geometry_df = geometry_df.dropDuplicates(["ISO3166-1-Alpha-3"])

In [0]:
geometry_df.write.mode("overwrite").saveAsTable("nasa_eonet.events_gold.dim_geometry")

fact_events

In [0]:
events_df = events_df.drop('geometry', 'avg_point', 'country_geom')

In [0]:
events_df.write.format("delta").mode('overwrite').saveAsTable("nasa_eonet.events_gold.fact_events")